In [1]:
import numpy as np
import pandas as pd
import requests 
import time 
import json
import os
from tqdm import tqdm
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'

In [ ]:
TOKEN = 'please-place-your-own-token'
headers = {'accept': 'application/json', 'Authorization': TOKEN}

# birds

In [3]:
birds_information = pd.read_csv(RESULTS / 'species-range-rasters' / 'birds_bounding_box.csv')

In [4]:
failed_sisid = []

In [5]:
failed_sisid

[]

In [ ]:
for i in tqdm(birds_information.index):
    try: 
        sisid = birds_information.loc[i, 'sisid']
        url_1 = f'https://api.iucnredlist.org/api/v4/taxa/sis/{sisid}'
        response_1 = requests.get(url_1, headers=headers)
        data_1 = response_1.json()
        assessment_id = data_1['assessments'][0]['assessment_id']
        birds_information.loc[i, 'assessment_id'] = assessment_id
        time.sleep(1)
        url_2 = f'https://api.iucnredlist.org/api/v4/assessment/{assessment_id}'
        response_2 = requests.get(url_2, headers=headers)
        assessment_details = response_2.json()
        time.sleep(1)
        with open(RESULTS / 'iucn-assessments' / 'birds' / f'{assessment_id}.json', 'w', encoding='utf-8') as f:
            json.dump(assessment_details, f, ensure_ascii=False, indent=2)
        

    except:
        print(f'{sisid} failed ...')
        failed_sisid.append(sisid)

100%|██████████| 10364/10364 [11:53:01<00:00,  4.13s/it] 


In [ ]:
birds_information[['sisid', 'name', 'assessment_id']].to_csv(RESULTS / 'iucn-assessments' / 'birds-sisid.csv', index=False)

# amphibians

In [ ]:
amphibians_information = pd.read_csv(RESULTS / 'iucn-assessments' / 'amphibians-sisid.csv')

In [56]:
failed_sisid_amphibians = []

In [ ]:
for i in tqdm(amphibians_information.index):
    try: 
        sisid = amphibians_information.loc[i, 'id_no']
        url_1 = f'https://api.iucnredlist.org/api/v4/taxa/sis/{sisid}'
        response_1 = requests.get(url_1, headers=headers)
        data_1 = response_1.json()
        assessment_id = data_1['assessments'][0]['assessment_id']
        amphibians_information.loc[i, 'assessment_id'] = assessment_id
        time.sleep(1)
        url_2 = f'https://api.iucnredlist.org/api/v4/assessment/{assessment_id}'
        response_2 = requests.get(url_2, headers=headers)
        assessment_details = response_2.json()
        time.sleep(1)
        with open(RESULTS / 'iucn-assessments' / 'amphibians' / f'{assessment_id}.json', 'w', encoding='utf-8') as f:
            json.dump(assessment_details, f, ensure_ascii=False, indent=2)
        
        amphibians_information.to_csv(RESULTS / 'iucn-assessments' / 'amphibians-sisid.csv', index=False)
        

    except:
        print(f'{sisid} failed ...')
        failed_sisid_amphibians.append(sisid)

100%|██████████| 2540/2540 [2:58:33<00:00,  4.22s/it]  


# mammals

In [ ]:
mammals_information = pd.read_csv(RESULTS / 'iucn-assessments' / 'mammals-sisid.csv')

In [13]:
failed_sisid_mammals = []

In [ ]:
for i in tqdm(mammals_information.index):
    try: 
        sisid = mammals_information.loc[i, 'id_no']
        url_1 = f'https://api.iucnredlist.org/api/v4/taxa/sis/{sisid}'
        response_1 = requests.get(url_1, headers=headers)
        data_1 = response_1.json()
        assessment_id = data_1['assessments'][0]['assessment_id']
        mammals_information.loc[i, 'assessment_id'] = assessment_id
        time.sleep(1)
        url_2 = f'https://api.iucnredlist.org/api/v4/assessment/{assessment_id}'
        response_2 = requests.get(url_2, headers=headers)
        assessment_details = response_2.json()
        time.sleep(1)
        with open(RESULTS / 'iucn-assessments' / 'mammals' / f'{assessment_id}.json', 'w', encoding='utf-8') as f:
            json.dump(assessment_details, f, ensure_ascii=False, indent=2)
        
        mammals_information.to_csv(RESULTS / 'iucn-assessments' / 'mammals-sisid.csv', index=False)
        

    except:
        print(f'{sisid} failed ...')
        failed_sisid_mammals.append(sisid)

100%|██████████| 5/5 [00:19<00:00,  3.91s/it]


# reptiles

In [ ]:
reptiles_information = pd.read_csv(RESULTS / 'iucn-assessments' / 'reptiles-sisid.csv')

In [10]:
failed_sisid_reptiles = []

In [ ]:
for i in tqdm(reptiles_information.index):
    try: 
        sisid = reptiles_information.loc[i, 'id_no']
        url_1 = f'https://api.iucnredlist.org/api/v4/taxa/sis/{sisid}'
        response_1 = requests.get(url_1, headers=headers)
        data_1 = response_1.json()
        assessment_id = data_1['assessments'][0]['assessment_id']
        reptiles_information.loc[i, 'assessment_id'] = assessment_id
        time.sleep(1)
        url_2 = f'https://api.iucnredlist.org/api/v4/assessment/{assessment_id}'
        response_2 = requests.get(url_2, headers=headers)
        assessment_details = response_2.json()
        time.sleep(1)
        with open(RESULTS / 'iucn-assessments' / 'reptiles' / f'{assessment_id}.json', 'w', encoding='utf-8') as f:
            json.dump(assessment_details, f, ensure_ascii=False, indent=2)
        
        reptiles_information.to_csv(RESULTS / 'iucn-assessments' / 'reptiles-sisid.csv', index=False)
        

    except:
        print(f'{sisid} failed ...')
        failed_sisid_reptiles.append(sisid)

100%|██████████| 5/5 [00:18<00:00,  3.78s/it]


---
## Compile assessment reports

Parse the downloaded IUCN JSONs into habitat-preference, elevation-range and Red-List-category tables (former notebook `3-compile-assessment-reports`).

In [1]:
import pandas as pd
import json
import os
from tqdm import tqdm

In [5]:
def process_a_taxon(taxon):
    assessments_taxon = os.listdir(RESULTS / 'iucn-assessments' / taxon) 

    habitat_preferences = pd.DataFrame([])
    elevation_range = pd.DataFrame([])
    red_list_category = pd.DataFrame([])
    i,j = 0, 0 

    for a_name in tqdm(assessments_taxon):
        a_path = RESULTS / 'iucn-assessments' / taxon / a_name 
        # print(a_path)
        with open(a_path, 'r', encoding='utf-8') as f:
            assessment = json.load(f)
        
        sisid = assessment['sis_taxon_id']
        assessment_id = assessment['assessment_id']
        scientific_name = assessment['taxon']['scientific_name']
        the_category = assessment['red_list_category']['code']

        for m in range(len(assessment['habitats'])):
            a_habitat_info = assessment['habitats'][m]
            habitat_preferences.loc[i, 'sisid'] = sisid 
            habitat_preferences.loc[i, 'assessment_id'] = assessment_id
            habitat_preferences.loc[i, 'scientific_name'] = scientific_name
            habitat_preferences.loc[i, 'majorImportance'] = a_habitat_info['majorImportance']
            habitat_preferences.loc[i, 'season'] = a_habitat_info['season'] 
            habitat_preferences.loc[i, 'suitability'] = a_habitat_info['suitability'] 
            habitat_preferences.loc[i, 'description'] = a_habitat_info['description']['en']
            habitat_preferences.loc[i, 'code'] = a_habitat_info['code']
            i += 1
       

        elevation_range.loc[j, 'sisid'] = sisid 
        elevation_range.loc[j, 'assessment_id'] = assessment_id 
        elevation_range.loc[j, 'scientific_name'] = scientific_name 
        elevation_range.loc[j, 'lower'] = assessment['supplementary_info']['lower_elevation_limit']
        elevation_range.loc[j, 'upper'] = assessment['supplementary_info']['upper_elevation_limit']

        red_list_category.loc[j, 'sisid'] = sisid 
        red_list_category.loc[j, 'scientific_name'] = scientific_name 
        red_list_category.loc[j, 'category'] = the_category

        j += 1
    
    return habitat_preferences, elevation_range, red_list_category

In [ ]:
habitat_preferences_amphibians, elevation_range_amphibians, _ = process_a_taxon('amphibians')

100%|██████████| 7640/7640 [01:57<00:00, 64.85it/s] 


In [6]:
habitat_preferences_birds, elevation_range_birds, red_list_category_birds = process_a_taxon('birds')

100%|██████████| 10374/10374 [03:03<00:00, 56.39it/s]


In [ ]:
habitat_preferences_mammals, elevation_range_mammals, _ = process_a_taxon('mammals')

100%|██████████| 5750/5750 [01:02<00:00, 92.25it/s] 


In [ ]:
habitat_preferences_reptiles, elevation_range_reptiles, _ = process_a_taxon('reptiles')

100%|██████████| 9870/9870 [01:50<00:00, 89.70it/s] 


In [10]:
habitat_preferences_amphibians.to_csv(RESULTS / 'iucn-assessments' / 'habitat-preferences-amphibians.csv', index=False)
habitat_preferences_birds.to_csv(RESULTS / 'iucn-assessments' / 'habitat-preferences-birds.csv', index=False)
habitat_preferences_mammals.to_csv(RESULTS / 'iucn-assessments' / 'habitat-preferences-mammals.csv', index=False)
habitat_preferences_reptiles.to_csv(RESULTS / 'iucn-assessments' / 'habitat-preferences-reptiles.csv', index=False)

In [11]:
elevation_range_amphibians.to_csv(RESULTS / 'iucn-assessments' / 'elevation-range-amphibians.csv', index=False)
elevation_range_birds.to_csv(RESULTS / 'iucn-assessments' / 'elevation-range-birds.csv', index=False)
elevation_range_mammals.to_csv(RESULTS / 'iucn-assessments' / 'elevation-range-mammals.csv', index=False)
elevation_range_reptiles.to_csv(RESULTS / 'iucn-assessments' / 'elevation-range-reptiles.csv', index=False)

In [8]:
red_list_category_birds.to_csv(RESULTS / 'iucn-assessments' / 'redlist-category-birds.csv', index=False)